# 🧹 Data Cleaning & Preparation
**Task 1 — Data Science Internship**  
**Author:** Affan Inamdar  
**Date:** 2026

---
## Objective
Clean and prepare a raw dataset for analysis by:
- Identifying and handling **missing values**
- Removing **duplicate records**
- Fixing **incorrect data types**
- Standardising **inconsistent values**
- Documenting every change made

---

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries imported successfully')

## Step 2 — Load Raw Dataset

In [ ]:
# Load the raw dataset
df_raw = pd.read_csv('../data/raw_data.csv')

# Always keep a backup of original data — never modify raw
df = df_raw.copy()

print(f'✅ Dataset loaded successfully')
print(f'📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns')

## Step 3 — Initial Data Exploration (Before Cleaning)

In [ ]:
# First look at the data
print('=== FIRST 5 ROWS ===')
df.head()

In [ ]:
# Last 5 rows
print('=== LAST 5 ROWS ===')
df.tail()

In [ ]:
# Dataset shape and data types
print('=== DATASET INFO ===')
df.info()

In [ ]:
# Statistical summary
print('=== STATISTICAL SUMMARY ===')
df.describe(include='all')

In [ ]:
# Column names list
print('=== COLUMN NAMES ===')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

## Step 4 — Identify Missing Values

In [ ]:
# Count missing values in each column
missing_count  = df.isnull().sum()
missing_pct    = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Column'        : df.columns,
    'Missing Count' : missing_count.values,
    'Missing %'     : missing_pct.values,
    'Data Type'     : df.dtypes.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_df) == 0:
    print('✅ No missing values found!')
else:
    print(f'⚠️  Found missing values in {len(missing_df)} columns:')
    print(missing_df.to_string(index=False))

In [ ]:
# Visualise missing values heatmap
if df.isnull().sum().sum() > 0:
    plt.figure(figsize=(14, 6))
    sns.heatmap(df.isnull(), yticklabels=False, cbar=True,
                cmap='Reds', linewidths=0.1)
    plt.title('Missing Values Heatmap (Red = Missing)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../screenshots/missing_values_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('📸 Screenshot saved!')
else:
    print('No missing values to visualise.')

## Step 5 — Handle Missing Values

In [ ]:
# Record count before
rows_before = len(df)
print(f'Rows before handling missing values: {rows_before}')

# ── Strategy 1: Fill numeric columns with MEDIAN ──
# (Median is better than Mean — not affected by outliers)
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  ✅ [{col}] → filled {df[col].isnull().sum()} NaN with median = {median_val:.2f}')

# ── Strategy 2: Fill categorical columns with MODE ──
# (Mode = most frequent value)
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f'  ✅ [{col}] → filled NaN with mode = "{mode_val}"')

# ── Strategy 3: Drop rows where ALL values are missing ──
df.dropna(how='all', inplace=True)

print(f'\nRows after handling missing values: {len(df)}')
print(f'Total missing values remaining: {df.isnull().sum().sum()}')

## Step 6 — Identify and Remove Duplicate Records

In [ ]:
# Check for duplicates
dup_count = df.duplicated().sum()
print(f'🔍 Total duplicate rows found: {dup_count}')

if dup_count > 0:
    print('\nSample duplicate rows:')
    print(df[df.duplicated(keep=False)].head(10))

    # Remove duplicates — keep first occurrence
    df.drop_duplicates(keep='first', inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'\n✅ Duplicates removed. Rows remaining: {len(df)}')
else:
    print('✅ No duplicate rows found!')

## Step 7 — Fix Incorrect Data Types

In [ ]:
# Check current data types
print('=== CURRENT DATA TYPES ===')
print(df.dtypes)

# ── Fix Date columns ──
# Look for columns that contain 'date' in name but are stored as object
date_cols = [col for col in df.columns if 'date' in col.lower() or 'Date' in col]
for col in date_cols:
    if df[col].dtype == 'object':
        try:
            df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')
            print(f'  ✅ [{col}] converted from object → datetime')
        except Exception as e:
            print(f'  ⚠️  [{col}] could not convert: {e}')

# ── Fix numeric columns stored as strings ──
for col in df.columns:
    if df[col].dtype == 'object':
        # Try converting to numeric — if most values work, convert
        converted = pd.to_numeric(df[col].str.replace(',','').str.replace('₹','').str.strip(),
                                  errors='coerce')
        non_null_pct = converted.notna().sum() / len(df)
        if non_null_pct > 0.85:  # 85%+ values are numeric
            df[col] = converted
            print(f'  ✅ [{col}] converted from object → numeric ({non_null_pct*100:.0f}% success)')

print('\n=== UPDATED DATA TYPES ===')
print(df.dtypes)

## Step 8 — Fix Inconsistent Values

In [ ]:
# ── Check all categorical columns for inconsistencies ──
cat_columns = df.select_dtypes(include='object').columns

print('=== UNIQUE VALUES IN CATEGORICAL COLUMNS ===')
for col in cat_columns:
    unique_vals = df[col].unique()
    print(f'\n[{col}] — {len(unique_vals)} unique values:')
    print(f'  {sorted([str(v) for v in unique_vals])}')

In [ ]:
# ── Fix 1: Standardise text — strip whitespace, fix capitalisation ──
for col in cat_columns:
    original_unique = df[col].nunique()
    df[col] = df[col].astype(str).str.strip()        # remove leading/trailing spaces
    df[col] = df[col].str.title()                     # Title Case: "mumbai" → "Mumbai"
    new_unique = df[col].nunique()
    if original_unique != new_unique:
        print(f'  ✅ [{col}] unique values: {original_unique} → {new_unique} (after standardisation)')

print('\n✅ Text standardisation complete')

In [ ]:
# ── Fix 2: Standardise Gender column if it exists ──
gender_cols = [col for col in df.columns if 'gender' in col.lower() or 'sex' in col.lower()]
for col in gender_cols:
    print(f'Before [{col}]:', df[col].value_counts().to_dict())
    df[col] = df[col].str.upper().str.strip()
    df[col] = df[col].replace({
        'M': 'Male', 'F': 'Female',
        'MALE': 'Male', 'FEMALE': 'Female',
        'male': 'Male', 'female': 'Female',
        '1': 'Male', '0': 'Female'
    })
    print(f'After  [{col}]:', df[col].value_counts().to_dict())

In [ ]:
# ── Fix 3: Handle outliers in numeric columns ──
# Using IQR method — standard approach
print('=== OUTLIER DETECTION (IQR METHOD) ===')

numeric_cols = df.select_dtypes(include=['int64','float64']).columns
outlier_summary = []

for col in numeric_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers    = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    if len(outliers) > 0:
        outlier_summary.append({
            'Column': col, 'Outliers': len(outliers),
            'Lower Bound': round(lower_bound, 2),
            'Upper Bound': round(upper_bound, 2)
        })

if outlier_summary:
    print(pd.DataFrame(outlier_summary).to_string(index=False))

    # Cap outliers to bounds (Winsorization) — better than dropping rows
    for item in outlier_summary:
        col = item['Column']
        df[col] = df[col].clip(lower=item['Lower Bound'], upper=item['Upper Bound'])
        print(f'  ✅ [{col}] outliers capped to [{item["Lower Bound"]}, {item["Upper Bound"]}]')
else:
    print('✅ No significant outliers found!')

In [ ]:
# ── Fix 4: Remove negative values from columns that can't be negative ──
non_negative_cols = [col for col in df.columns
                     if any(w in col.lower() for w in
                     ['salary','price','amount','quantity','qty','revenue','sales','age'])]

for col in non_negative_cols:
    if col in df.columns and df[col].dtype in ['int64','float64']:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            df[col] = df[col].abs()  # convert negative to positive
            print(f'  ✅ [{col}] fixed {neg_count} negative values → converted to positive')

## Step 9 — Add Useful Derived Columns

In [ ]:
# Add derived columns that will be useful for analysis

# ── Age Group ──
if 'Age' in df.columns:
    df['Age_Group'] = pd.cut(df['Age'],
                             bins=[0, 25, 35, 45, 55, 100],
                             labels=['18-25','26-35','36-45','46-55','55+'])
    print('✅ Age_Group column added')

# ── Salary Band ──
if 'Salary' in df.columns:
    df['Salary_Band'] = pd.cut(df['Salary'],
                               bins=[0, 500000, 1000000, 1500000, 2000000, 999999999],
                               labels=['< 5L', '5L-10L', '10L-15L', '15L-20L', '> 20L'])
    print('✅ Salary_Band column added')

# ── Year and Month from dates ──
for col in df.columns:
    if df[col].dtype == 'datetime64[ns]':
        base_name = col.replace('_Date','').replace('Date','')
        df[f'{base_name}_Year']  = df[col].dt.year
        df[f'{base_name}_Month'] = df[col].dt.month_name()
        df[f'{base_name}_Quarter'] = 'Q' + df[col].dt.quarter.astype(str)
        print(f'✅ Year/Month/Quarter extracted from [{col}]')

print(f'\n✅ Dataset now has {df.shape[1]} columns')

## Step 10 — Final Validation & Quality Report

In [ ]:
print('=' * 55)
print('       DATA CLEANING SUMMARY REPORT')
print('=' * 55)
print(f'  Original rows         : {rows_before}')
print(f'  Final rows            : {len(df)}')
print(f'  Rows removed          : {rows_before - len(df)}')
print(f'  Final columns         : {df.shape[1]}')
print(f'  Missing values left   : {df.isnull().sum().sum()}')
print(f'  Duplicate rows left   : {df.duplicated().sum()}')
print('=' * 55)

print('\n=== FINAL DATA TYPES ===')
print(df.dtypes)

print('\n=== FINAL SAMPLE (5 rows) ===')
df.head()

In [ ]:
# ── Visualise cleaned data ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Data Quality Dashboard — After Cleaning', fontsize=16, fontweight='bold')

# Plot 1: Missing values after cleaning
axes[0,0].bar(['Missing Values'], [df.isnull().sum().sum()], color='#22c55e')
axes[0,0].set_title('Missing Values (After Cleaning)')
axes[0,0].set_ylabel('Count')

# Plot 2: Numeric distribution (first numeric column)
num_col = df.select_dtypes(include='number').columns
if len(num_col) > 0:
    axes[0,1].hist(df[num_col[0]].dropna(), bins=30, color='#f97316', edgecolor='white')
    axes[0,1].set_title(f'Distribution of {num_col[0]}')
    axes[0,1].set_xlabel(num_col[0])

# Plot 3: Category counts (first categorical column)
cat_col = df.select_dtypes(include='object').columns
if len(cat_col) > 0:
    vc = df[cat_col[0]].value_counts().head(8)
    axes[1,0].barh(vc.index, vc.values, color='#3b82f6')
    axes[1,0].set_title(f'Top Values: {cat_col[0]}')
    axes[1,0].set_xlabel('Count')

# Plot 4: Data types breakdown
type_counts = df.dtypes.astype(str).value_counts()
axes[1,1].pie(type_counts.values, labels=type_counts.index,
              autopct='%1.1f%%', colors=['#f97316','#22c55e','#3b82f6','#a855f7'])
axes[1,1].set_title('Data Types Distribution')

plt.tight_layout()
plt.savefig('../screenshots/cleaning_summary_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 Summary chart saved!')

## Step 11 — Save Cleaned Dataset

In [ ]:
# Save cleaned dataset
df.to_csv('../data/cleaned_data.csv', index=False)

print('✅ Cleaned dataset saved to: ../data/cleaned_data.csv')
print(f'   Final shape: {df.shape[0]} rows × {df.shape[1]} columns')
print('\n🎉 Data Cleaning Complete!')